## Environment Setup

In [9]:
import os
import json
import xpd_tools
from tkinter.constants import N
import pandas as pd
import numpy as np

## AGENT - Initialization

### Initialize the Agent and set the logical paths and globals of our system.

For Bluesky:
- Queueserver toggle
- Tiled profile

Other:
- HTTP API and Server uri's
- ZMQ consumer addresses

Evaluation methods:
- xray, uvvis, or xray-uvis


In [ ]:
from xpd_tools.optimization.agent import BuildAgent

# Queueserver connection
HTTP_SERVER_URI = os.environ.get(
    "QSERVER_HTTP_URI", 
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov"
    )
HTTP_API_KEY = os.environ.get(
    "QSERVER_HTTP_SERVER_API_KEY", 
    ""
    )
ZMQ_CONSUMER_ADDR = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
    )

# Historical data path
#AGENT_DATA_PATH = "tmp/checkpoints/agent_halide_data.csv"
AGENT_DATA_PATH = None

# Tiled URI for evaluation function
TILED_URI = os.environ.get(
    "TILED_URI", 
    "https://tiled.nsls2.bnl.gov"
    )
TILED_PROFILE = os.environ.get(
    "TILED_PROFILE", 
    "xpd"
    )

# Acquisition plan name (must be registered on the queueserver)
ACQUISITION_PLAN_NAME = "xray_uvvis_acquire"

build_agent = BuildAgent(
    # Use queueserver for async
    queue_server = True,
    # Agent
    agent_data_path = AGENT_DATA_PATH,
    # Http 
    http_server_uri = HTTP_SERVER_URI,
    http_api_key = HTTP_API_KEY,
    # ZMQ
    zmq_consumer_address = ZMQ_CONSUMER_ADDR,
    # Tiled
    tiled_profile = TILED_PROFILE,
    # Evaluation Method
    evaluation_method = 'xray'
)


# Provide the agent some metadata
build_agent.set_metadata({
    "beamline": "28id2",
    "tags": ["qserver", "bluesky"],
    "comment": "Halide Synthesis Test"
    })
print(build_agent.metadata_string)

beamline: 28id2
tags: ['qserver', 'bluesky']
comment: Halide Synthesis Test



## BEAMLINE - PDF Xray Diffraction

Objective correlation function with the phases that will be tested.


Correlation functions:
- pearson, nn-matrix, weighted-profile-r, cross-correlation, or ensemble

In [ ]:
from xpd_tools.optimization.helpers.phases import Phase

# PDF correlation function
PDF_FUNCTION = "pearson"

# Set the phase selection objectives and directions
build_agent.set_xray_objectives(
    # Measurement
    max_retries     = 10,
    retry_delay     = 2.0,
    # Configuration
    exposure        = 600.0,
    frame_acq_time  = 1.5,
    no_dark         = False,
    stream_name     = "scattering",
    # Screening: UV-Vis is measured (screen_only/screen_and_record) or not
    # (unscreened). We configure full UV-Vis objectives below, so we want
    # the measurement recorded, not just used to gate acquisition.
    screening       = "screen_and_record",
    # Quality Checks
    use_good_bad    = True,
    good_target     = 2,
    max_bad         = 3,
    num_abs         = 16,
    num_flu         = 16,
    # PDF correlation masking window (Angstroms)
    min_radius      = 2.0,
    max_radius      = 20.0,
    # Phase Fitting
    objective_function = PDF_FUNCTION,
    phases   = [
        Phase(
            name      = "CsPbBr3",
            gr        = "refdata/CsPbBr3.gr",
            cif       = "refdata/CsPbBr3.cif",
            minimize  = False
            ),
        Phase(
            name      = "CsBr",
            gr        = "refdata/CsBr.gr",
            cif       = "refdata/CsBr.cif",
            minimize  = True
            ),
        Phase(
            name      = "Cs4PbBr6",
            gr        = "refdata/Cs4PbBr6.gr",
            cif       = "refdata/Cs4PbBr6.cif",
            minimize  = True
            ),
    ]
  )
print(build_agent.xray_settings)
print(build_agent.quality_policy)
print("screening:", build_agent.screening)
print("min/max radius:", build_agent.min_radius, build_agent.max_radius)

## BEAMLINE - UVvis Screening

In [12]:
# Optimization Parameters
PEAK_TARGET         = 450   # nm
PEAK_TOLERANCE      = 5     # nm

from xpd_tools.optimization.helpers.qepro import PlqyReference, SpectraFitSettings

build_agent.set_uvvis_objectives(
    # Objective Target
    peak_target             = PEAK_TARGET,
    peak_tolerance          = PEAK_TOLERANCE,
    max_retries             = 10,
    retry_delay             = 2.0,
    # Screening, data selection, and fitting windows
    fit_settings = SpectraFitSettings(
        pl_screen_key_height        = 200,
        pl_screen_peak_height       = 50,
        pl_screen_peak_distance     = 100,
        pl_percent_range            = (40, 100),
        pl_wavelength_range         = (400, 800),
        pl_fit_maxfev               = 100000,
        pl_fit_r2_window_sigma      = 3,
        absorbance_percent_range    = (10, 70),
        absorbance_wavelength_range = (210, 700),
    ),
    # Calibration Standard Reference (reference_type defaults to "quinine")
    plqy = PlqyReference(
        excitation_wavelength_nm   = 365,
        absorbance                 = 0.361,
        pl_integral                = 952628,
        refractive_index           = 1.337,
        plqy                       = 0.546,
        solvent_refractive_index   = 1.506,
    ),
)

print(build_agent.fit_settings)
print(build_agent.plqy)

SpectraFitSettings(pl_screen_key_height=200, pl_screen_peak_height=50, pl_screen_peak_distance=100, pl_percent_range=(40, 100), pl_wavelength_range=(400, 800), pl_fit_maxfev=100000, pl_fit_r2_window_sigma=3, absorbance_percent_range=(10, 70), absorbance_wavelength_range=(210, 700))
PlqyReference(reference_type='quinine', excitation_wavelength_nm=365, absorbance=0.361, pl_integral=952628, refractive_index=1.337, plqy=0.546, solvent_refractive_index=1.506)


## BLOP - DOFs

Degrees of freedom that BLOP can modify

In [ ]:
# Set the Agent DOFs
from xpd_tools.optimization.helpers.dofs import Pump

build_agent.set_dofs(
    pumps = [
      Pump(
        name = "CsPb",
        bounds = (10, 200),
        id = "dds2_p1"
      ),
      Pump(
        name = "Br",
        bounds = (5, 200),
        id = "dds2_p2"
      ),
      Pump(
        name = "I2",
        bounds = (0, 200),
        id = "dds3_p1"
      )
    ]
  )
print(build_agent.dofs)

## BEAMLINE - Experimental Harware Parameters

In [ ]:
from xpd_tools.optimization.plans import FlowSource, DilutionStage, WashCycle

build_agent.experiment(
    sources = [
        FlowSource(
            dof="infusion_rate_CsPb",
            pump='dds2_p1',
            precursor="CsPbOA",
            sample_label="CsPb",
        ),
        FlowSource(
            dof="infusion_rate_Br",
            pump='dds2_p2',
            precursor="TOABr",
            sample_label="Br",
        ),
        FlowSource(
            dof="infusion_rate_I2",
            pump='dds3_p1',
            precursor="ZnI2",
            sample_label="I2",
        ),
    ],
    dilutions = [
        DilutionStage(
            pump='dds1_p1',
            ratio=1.0,
            position="before_equilibrium",
            syringe_ml=20,
            material="plastic_BD",
            target_ml=20,
        ),
        DilutionStage(
            pump='ultra2',
            ratio=1.0,
            position="after_equilibrium",
            syringe_ml=100,
            material="steel",
            target_ml=100,
            wait_sec=30,
        ),
    ],
    wash_cycles=[
        WashCycle(pump='ultra1'),
    ]
)

## BLOP - Load and preprocess historical data

In [15]:
if AGENT_DATA_PATH is not None:
    df = pd.read_csv(AGENT_DATA_PATH, index_col=0)
    df = df[["Peak", "log_FWHM", "log_PLQY", "infusion_rate_CsPb", "infusion_rate_Br", "infusion_rate_Cl"]]
    df["peak_distance"] = (df["Peak"] - PEAK_TARGET).abs()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

## BLOP - Save / Load Config Check

In [ ]:
# Two methods, if a filename isn't provided, a json is returned
build_config = build_agent.to_config()

with open('autonomous_build_config.json', 'w', encoding='utf-8') as f:
    json.dump(build_config, f, ensure_ascii=False, indent=4)

with open('autonomous_build_config.json', 'r') as file:
    read_config = json.load(file)
# from_config is a classmethod -- it returns a new, fully-built agent rather
# than mutating an existing instance in place.
read_agent = BuildAgent.from_config(read_config, http_api_key=build_agent.http_api_key)

print(build_agent.__dict__)
print(read_agent.__dict__)

# BuildAgent has no __eq__, so compare configuration state via __dict__
# rather than `==` (which would just compare object identity).
assert build_agent.__dict__ == read_agent.__dict__, "Build and Read agents are not equal"


# If a filename is provided, a json is written to the current wdir
build_agent.to_config('autonomous_build_config.json')

with open('autonomous_build_config.json', 'r') as file:
    read_config = json.load(file)
read_agent = BuildAgent.from_config(read_config, http_api_key=build_agent.http_api_key)

assert build_agent.__dict__ == read_agent.__dict__, "Build and Read agents are not equal"


## BLOP - Build and Run Agent

In [ ]:
run_agent = build_agent.build()
run_agent.ax_client.configure_generation_strategy(
    initialize_with_center=False,
    # Loads historical data?
    use_existing_trials_for_initialization=True,
)


### Optional: stop early on success

Independent of the `iterations=` ceiling below -- a background thread polls
completed trials and calls `agent.stop()` as soon as either criterion is
met. `min_correlation` and the `max_fwhm`/`min_plqy` pair are separate,
either-or success paths, not combined with AND. Skip this cell entirely to
keep the plain `iterations=` behavior.

In [ ]:
build_agent.set_success_criteria(
    # Stop once any wanted phase (CsPbBr3 here) reaches this correlation.
    min_correlation = 0.9,
    poll_interval   = 5.0,
)
print(build_agent.success_criteria)

In [ ]:
from xpd_tools.optimization.stopping import watch_and_stop

# Each iteration measures a real X-ray exposure (600s, set above) plus
# UV-Vis screening -- tune ITERATIONS before actually running this cell.
ITERATIONS = 20

future = run_agent.run(iterations=ITERATIONS)

# Starts the background watcher from the success criteria configured
# above; it stops the campaign early on success, otherwise the
# iterations= ceiling above is what ends the campaign.
watcher_thread = watch_and_stop(run_agent, future, build_agent)

# Blocks until the campaign ends, whichever of the two stops it first.
future.result()

## Summarize and Export Data

In [ ]:
df = run_agent.ax_client.summarize()
df.to_csv("tmp/output/agent_halide_data.csv")